# Embedding Route

In [25]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()

CUDA error: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
, switching to CPU.
Loaded index 'uu'.


In [26]:
from utils.retriever.uu.embedding_uu import get_embedding_uu

retriever = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

In [27]:
retriever.invoke("jawa barat")

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Graph Route

In [28]:
from utils.llm import get_llm
from utils.config import get_config

config = get_config()
llm = get_llm("nvidia-llama3.1")

In [29]:
from utils.retriever.uu.graph_uu import get_graph_uu

retriever = get_graph_uu(llm=llm, config=config)

In [30]:
result = retriever.invoke("Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: '36', tahun: '1953', jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DICABUT]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[{'revokedRegulation': {'tentang': 'BANK TABUNGAN POS', 'tahun': '1953', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-36-tahun-1953', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '36', 'status': 'Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara'}, 'revokingRegulation': {'tentang': 'BANK TABUNGAN NEGARA', 'tahun': '1964', 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-2-tahun-1964', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': '2', 'status': 'Berlaku'}}]

> Finished chain.


In [31]:
result

{'query': 'Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?',
 'result': 'Pertanyaan Anda tentang Undang-Undang No 36 Tahun 1953 yang dicabut peraturan apa saja.\n\nDari hasil kueri Cypher, saya menemukan bahwa Undang-Undang No 36 Tahun 1953 dicabut oleh Undang-Undang No 2 Tahun 1964 tentang Bank Tabungan Negara.\n\nBerikut adalah informasi lebih lanjut tentang peraturan yang dicabut:\n\n* Nomor: Undang-Undang No 36 Tahun 1953\n* Jenis peraturan: Undang-Undang\n* Tahun: 1953\n* Tempat penetapan: Jakarta\n* Pemrakarsa: Pemerintah Pusat\n* Nomor: 36\n* Status: Tidak Berlaku (Dicabut oleh Undang-Undang No 2 Tahun 1964)\n\nPeraturan yang mencabut Undang-Undang No 36 Tahun 1953 adalah:\n\n* Nomor: Undang-Undang No 2 Tahun 1964\n* Jenis peraturan: Undang-Undang\n* Tahun: 1964\n* Tempat penetapan: Jakarta\n* Pemrakarsa: Pemerintah Pusat\n* Nomor: 2\n* Status: Berlaku\n\nPeraturan yang dicabut adalah Undang-Undang No 36 Tahun 1953 tentang Bank Tabungan Pos.'}

# All Together

In [9]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu

llm = get_llm("nvidia-llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [2]:
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

In [3]:
from utils.chain.rag import create_combined_answer_chain

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

In [4]:
result = chain_quality.invoke({"question": "Undang Undang No 36 Tahun 1953 dicabut peraturan apa saja?", "chat_history": ""})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (p:Peraturan {nomor: 36, tahun: 1953, jenis_peraturan: 'UNDANG-UNDANG'})
MATCH (p)-[:DICABUT]->(revoked:Peraturan)
RETURN 
  p AS revokedRegulation, 
  revoked AS revokingRegulation

Full Context:
[{'revokedRegulation': {'tentang': 'BANK TABUNGAN POS', 'tahun': 1953, 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-36-tahun-1953', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 36, 'status': 'Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara'}, 'revokingRegulation': {'tentang': 'BANK TABUNGAN NEGARA', 'tahun': 1964, 'jenis_peraturan': 'UNDANG-UNDANG', 'tempat_penetapan': 'Jakarta', 'id': 'https://peraturan.go.id/id/uu-no-2-tahun-1964', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 2, 'status': 'Berlaku'}}]

> Finished chain.


In [5]:
print(result['answer'])

Berikut adalah jawaban pertanyaan Anda:

Undang-undang No 36 Tahun 1953 dicabut oleh Undang-undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara. Peraturan yang dicabut tersebut adalah tentang BANK TABUNGAN POS.

Informasi lebih lanjut tentang peraturan yang dicabut dan mencabutnya adalah sebagai berikut:

* Peraturan yang dicabut: Undang-undang No 36 Tahun 1953
* Jenis peraturan: Undang-undang
* Tempat penetapan: Jakarta
* Pemrakarsa: Pemerintah Pusat
* Nomor: 36
* Status: Tidak Berlaku Dicabut Oleh :Undang-Undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara

* Peraturan yang mencabut: Undang-undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara
* Jenis peraturan: Undang-undang
* Tempat penetapan: Jakarta
* Pemrakarsa: Pemerintah Pusat
* Nomor: 2
* Status: Berlaku

**Sources**:

- [Undang-undang No 36 Tahun 1953](file_url)
- [Undang-undang Nomor 2 Tahun 1964 Tentang Bank Tabungan Negara](file_url)


In [6]:
result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})

In [7]:
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '58f2f3c8-d6c5-4fed-98c1-b98efa75969f', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam

In [8]:
print(result['answer'])

Berikut adalah jawaban yang komprehensif berdasarkan informasi dari sumber UU:

Ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.

### Referensi:
- Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat (https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf#page=4)

### Daftar Regulasi Unik:

- Undang-undang Nomor 10 Tahun 2023

Jawaban ini didasarkan pada informasi dari sumber UU yang menyebutkan ibu kota Provinsi Jawa Barat berkedudukan di Kota Bandung.


In [9]:
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '58f2f3c8-d6c5-4fed-98c1-b98efa75969f', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam